# 🔢 Notebook 05 — Embeddings & FAISS Vector Store
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the labelled dataset
- Build formatted text chunks (question + context + answer)
- Generate dense embeddings with `S-PubMedBert-MS-MARCO (biomedical domain)`
- Build a FAISS `IndexFlatL2` index
- Persist index + chunk mapping to disk
- Sanity-check retrieval with 5 test queries (top-3)
- Measure retrieval latency (KPI: < 500ms)

### 📋 Deliverables
- `data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss`
- `data/embeddings/faiss_index/chunk_mapping.csv`
- `data/embeddings/faiss_index/chunk_mapping.pkl`

---

## 1. Imports & Setup

In [1]:
import os
import pickle
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("✅ Imports successful")

D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Imports successful


## 2. Load Data & Build Text Chunks

Each chunk combines **question + context + answer** into a single
retrieval unit. Including the context (medical abstract) is critical
because it contains the evidence a RAG pipeline needs to ground its answers.

In [2]:
data_path = "../data/processed/pubmedqa_labelled.csv"

df = pd.read_csv(data_path)
print(f"✅ Loaded: {data_path}")
print(f"   Total rows: {len(df):,}")
print(f"   Columns: {list(df.columns)}")

# Validate required columns
required_columns = {"question", "context", "answer"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Drop rows with missing text
before = len(df)
df = df.dropna(subset=["question", "context", "answer"]).copy()
df["question"] = df["question"].astype(str).str.strip()
df["context"] = df["context"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df = df[(df["question"] != "") & (df["context"] != "") & (df["answer"] != "")]
df = df.reset_index(drop=True)
print(f"   Rows after cleaning: {len(df):,} (dropped {before - len(df)})")

✅ Loaded: ../data/processed/pubmedqa_labelled.csv
   Total rows: 10,000
   Columns: ['question', 'context', 'answer', 'category']
   Rows after cleaning: 10,000 (dropped 0)


In [3]:
# ── Reserve last 1,000 rows as TRUE held-out evaluation set ──────────────
# These rows will NOT be added to the FAISS index, so RAG evaluation
# in NB 08 measures real generalisation rather than retrieval-cache effects.
# 1,000 rows gives a statistically robust holdout (was 200).

EVAL_HOLDOUT_SIZE = 1000

df_eval  = df.tail(EVAL_HOLDOUT_SIZE).copy().reset_index(drop=True)
df_index = df.iloc[:-EVAL_HOLDOUT_SIZE].copy().reset_index(drop=True)

print(f"📊 Dataset split:")
print(f"   Indexed (FAISS): {len(df_index):,} rows")
print(f"   Held-out (eval): {len(df_eval):,} rows")
print(f"   Total:           {len(df):,} rows")

# Save holdout for notebook 08
holdout_path = Path('../data/processed/eval_holdout.csv')
df_eval.to_csv(holdout_path, index=False)
print(f"\n✅ Holdout saved: {holdout_path}")
print(f"   Category distribution:")
print(df_eval['category'].value_counts().to_string())


📊 Dataset split:
   Indexed (FAISS): 9,000 rows
   Held-out (eval): 1,000 rows
   Total:           10,000 rows

✅ Holdout saved: ..\data\processed\eval_holdout.csv
   Category distribution:
category
Treatment     353
Diagnosis     190
Medication    167
General       135
Prevention    105
Symptoms       50


## 3. Generate Embeddings

Using `sentence-transformers/S-PubMedBert-MS-MARCO (biomedical domain)`:
- 384-dimensional embeddings
- Fast inference, good semantic quality
- FAISS requires `float32` arrays

In [4]:
# Build text chunks: question + context + answer
df_index["text_chunk"] = (
    "Question: " + df_index["question"] + "\n"
    + "Context: " + df_index["context"] + "\n"
    + "Answer: " + df_index["answer"]
)

text_chunks = df_index["text_chunk"].tolist()
n_chunks = len(text_chunks)

print(f"✅ Built {n_chunks:,} text chunks")
print(f"   Sample:\n{text_chunks[0][:300]}")

✅ Built 9,000 text chunks
   Sample:
Question: Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?
Context: Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compare


In [5]:
model_name = "pritamdeka/S-PubMedBert-MS-MARCO"  # Upgraded: biomedical domain model
model = SentenceTransformer(model_name)
print(f"✅ Loaded model: {model_name}")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

# %%
print(f"⏳ Encoding {n_chunks:,} chunks...")

start_time = time.time()

embeddings = model.encode(
    text_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # cosine similarity via dot product
)

encoding_time = time.time() - start_time

# FAISS requires float32
embeddings = np.asarray(embeddings, dtype=np.float32)

print(f"\n✅ Encoding complete!")
print(f"   Shape: {embeddings.shape}")
print(f"   Dtype: {embeddings.dtype}")
print(f"   Time: {encoding_time:.1f}s ({encoding_time/n_chunks*1000:.1f}ms per chunk)")

D:\Projects\Healthcare-RAG-Powered-Medical-QA-Assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Loaded model: pritamdeka/S-PubMedBert-MS-MARCO
   Embedding dimension: 768
⏳ Encoding 9,000 chunks...


Batches:   0%|          | 0/141 [00:00<?, ?it/s]


✅ Encoding complete!
   Shape: (9000, 768)
   Dtype: float32
   Time: 96.3s (10.7ms per chunk)


## 4. Build FAISS Index

`IndexFlatL2` — exact nearest-neighbor search using L2 distance.
- No training needed
- Dimension inferred from embeddings
- Lower distance = higher similarity

In [6]:
d = embeddings.shape[1]

index = faiss.IndexFlatL2(d)
index.add(embeddings)

print(f"✅ FAISS index built")
print(f"   Dimension: {d}")
print(f"   Total vectors: {index.ntotal:,}")

✅ FAISS index built
   Dimension: 768
   Total vectors: 9,000


## 5. Save Index & Chunk Mapping

In [7]:
output_dir = Path("../data/embeddings/faiss_index")
output_dir.mkdir(parents=True, exist_ok=True)

index_path = output_dir / "pubmedqa_index_flatl2.faiss"
mapping_csv_path = output_dir / "chunk_mapping.csv"
mapping_pkl_path = output_dir / "chunk_mapping.pkl"

# Save FAISS index
faiss.write_index(index, str(index_path))

# Save mapping table (chunk_id → question, context, answer, category, text_chunk)
# IMPORTANT: chunk_id values stay 0..(n_chunks-1) — they are positions in this FAISS index,
# NOT row numbers from the original labelled CSV. The eval set is held out separately.
mapping_df = df_index[["question", "context", "answer", "category", "text_chunk"]].copy()
mapping_df.insert(0, "chunk_id", np.arange(len(mapping_df), dtype=np.int32))

mapping_df.to_csv(mapping_csv_path, index=False)

with open(mapping_pkl_path, "wb") as f:
    pickle.dump(mapping_df, f)

print(f"✅ Saved FAISS index to: {index_path}")
print(f"   Vectors:   {index.ntotal:,}")
print(f"   File size: {os.path.getsize(index_path) / 1024**2:.1f} MB")
print(f"✅ Saved chunk mapping CSV to: {mapping_csv_path}")
print(f"✅ Saved chunk mapping PKL to: {mapping_pkl_path}")

# Final sanity: chunk count == FAISS vectors == df_index rows
assert index.ntotal == len(mapping_df) == len(df_index), \
    f"Mismatch: index={index.ntotal}, mapping={len(mapping_df)}, df_index={len(df_index)}"
print(f"✅ Sanity check passed: {index.ntotal:,} vectors == {len(mapping_df):,} mapping rows")

✅ Saved FAISS index to: ..\data\embeddings\faiss_index\pubmedqa_index_flatl2.faiss
   Vectors:   9,000
   File size: 26.4 MB
✅ Saved chunk mapping CSV to: ..\data\embeddings\faiss_index\chunk_mapping.csv
✅ Saved chunk mapping PKL to: ..\data\embeddings\faiss_index\chunk_mapping.pkl
✅ Sanity check passed: 9,000 vectors == 9,000 mapping rows


In [8]:
# ── Auto-upload to HuggingFace ────────────────────────────────────────────
import sys
sys.path.append(os.path.abspath('..'))

from src.data.hub import upload_file

print("\n📤 Syncing to HuggingFace...")

# FAISS index + chunk mapping (rebuilt from 9,800 indexed rows)
upload_file(
    "data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss",
    "embeddings/pubmedqa_index_flatl2.faiss"
)
upload_file(
    "data/embeddings/faiss_index/chunk_mapping.pkl",
    "embeddings/chunk_mapping.pkl"
)
upload_file(
    "data/embeddings/faiss_index/chunk_mapping.csv",
    "embeddings/chunk_mapping.csv"
)

# NEW: held-out eval set (200 rows NOT in the FAISS index)
upload_file(
    "data/processed/pubmedqa_eval_holdout.csv",
    "processed/pubmedqa_eval_holdout.csv"
)


📤 Syncing to HuggingFace...
  ❌ Failed: data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss — 401 Client Error. (Request ID: Root=1-6a0c1df4-3480fb11629d816030e17fe3;473b383f-5c25-49ba-9d83-599d7dcb32d7)

Repository Not Found for url: https://huggingface.co/api/datasets/AbdoMatrix/healthcare-rag-data/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated.
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.
  ❌ Failed: data/embeddings/faiss_index/chunk_mapping.pkl — 401 Client Error. (Request ID: Root=1-6a0c1df4-48be5f3b486c99d47e0dbac9;aec834c4-908a-4d97-87a0-b6d132b44fe6)

Repository Not Found for url: https://huggingface.co/api/datasets/AbdoMatrix/healthcare-rag-data/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If yo

False

## 6. Sanity-Check Retrieval (Top-3) + Latency Measurement

**M2 KPI: FAISS returns results in < 500ms**

In [9]:
test_queries = [
    "What are effective treatments for irritable bowel syndrome symptoms?",
    "Can hypothyroidism increase risk of fatty liver disease?",
    "Is laparoscopic prostatectomy superior to retropubic surgery?",
    "How accurate is diagnosis of acute otitis media in primary care?",
    "Does secondary isoniazid therapy reduce recurrent tuberculosis in HIV patients?",
]

# Encode queries
query_embeddings = model.encode(
    test_queries,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

k = 3  # top-3 results

# ── Run retrieval with timing ────────────────────────────────────────────
latencies = []

for qi, query in enumerate(test_queries):
    # Time only the FAISS search (not encoding)
    start = time.time()
    D, I = index.search(query_embeddings[qi:qi+1], k)
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)

    print("=" * 110)
    print(f"Query {qi + 1}: {query}")
    print(f"⏱️  Retrieval latency: {elapsed_ms:.2f}ms")
    print()

    for rank in range(k):
        idx = int(I[0, rank])
        dist = float(D[0, rank])
        chunk = mapping_df.loc[idx, "text_chunk"]

        print(f"  Top {rank + 1} | Chunk ID: {idx} | L2 Distance: {dist:.4f}")
        print(f"  {chunk[:300]}")
        if len(chunk) > 300:
            print("  ... [truncated]")
        print("-" * 110)

# ── Latency summary ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("⏱️  LATENCY SUMMARY")
print("=" * 60)
print(f"  Min:    {min(latencies):.2f}ms")
print(f"  Max:    {max(latencies):.2f}ms")
print(f"  Mean:   {np.mean(latencies):.2f}ms")
print(f"  Median: {np.median(latencies):.2f}ms")
print(f"\n  Index size: {index.ntotal:,} vectors (held-out: {EVAL_HOLDOUT_SIZE} rows)")
print()

if max(latencies) < 500:
    print("✅ KPI MET: All queries returned in < 500ms")
else:
    print("⚠️  KPI NOT MET: Some queries exceeded 500ms")

Query 1: What are effective treatments for irritable bowel syndrome symptoms?
⏱️  Retrieval latency: 7.92ms

  Top 1 | Chunk ID: 236 | L2 Distance: 192.1368
  Question: Histamine 1 receptor antagonist in symptomatic treatment of renal colic accompanied by nausea: two birds with one stone?
Context: The pain-relieving efficacy of antagonists of histamine 1 (H1) receptors that are widely found in the ureter and that cause contractions in renal colic was pres
  ... [truncated]
--------------------------------------------------------------------------------------------------------------
  Top 2 | Chunk ID: 8172 | L2 Distance: 192.1646
  Question: Is there any food I can eat?
Context: Inflammatory bowel disease (Crohn's disease and ulcerative colitis) and irritable bowel syndrome are chronic, debilitating gastrointestinal (GI) disorders. There are no known causes of inflammatory bowel disease and/or irritable bowel syndrome. Both of
  ... [truncated]
-----------------------------------------

## ✅ Summary

| Item | Result |
|---|---|
| Text chunks | Question + Context + Answer |
| Embedding model | `S-PubMedBert-MS-MARCO (biomedical domain)` (384d) |
| FAISS index type | `IndexFlatL2` |
| Vectors stored | {n_chunks:,} |
| Sanity check | 5 queries × top-3 results |
| Latency KPI | < 500ms per query |

### Files Saved
- `data/embeddings/faiss_index/pubmedqa_index_flatl2.faiss`
- `data/embeddings/faiss_index/chunk_mapping.csv`
- `data/embeddings/faiss_index/chunk_mapping.pkl`

---

### ➡️ Next Step
Open **`05_rag_pipeline.ipynb`** to build the LangChain RAG pipeline.